# Data Management

Companion notebook for **[Data Management](https://jayantkhd.vercel.app/blog/gen-ai-zero-to-one/data-management)** — *Gen AI: Zero to One · Phase 0*.

Load, stream, convert, split, and cache datasets with a repeatable workflow.

## Step 1 — Install

In [ ]:
%pip install -q datasets huggingface_hub

## Step 2 — Load a dataset

Downloads IMDB the first time, then loads from `~/.cache/huggingface/datasets/`.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)
print(dataset["train"][0])

## Step 3 — Stream a large dataset

`streaming=True` returns an `IterableDataset` — rows arrive on demand, memory stays constant.

In [ ]:
stream = load_dataset(
    "wikimedia/wikipedia", "20220301.en", split="train", streaming=True
)

for i, example in enumerate(stream):
    print(example["title"])
    if i >= 4:
        break

## Step 4 — Convert formats

`datasets` uses Apache Arrow in memory. Parquet is the best on-disk format for ML.

In [ ]:
train = load_dataset("imdb", split="train")

train.to_csv("imdb_train.csv")
train.to_json("imdb_train.json")
train.to_parquet("imdb_train.parquet")

import os
for f in ("imdb_train.csv", "imdb_train.parquet"):
    print(f, round(os.path.getsize(f) / 1e6, 1), "MB")

## Step 5 — Reproducible splits

Always pass a `seed`. The same seed produces the same split every time.

In [ ]:
from data_utils import make_splits

train_ds, val_ds, test_ds = make_splits(load_dataset("imdb", split="train"))
print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

## Step 6 — Download and cache a model

Models cache to `~/.cache/huggingface/hub/` and load instantly afterwards.

In [ ]:
from huggingface_hub import hf_hub_download

path = hf_hub_download(
    repo_id="sentence-transformers/all-MiniLM-L6-v2",
    filename="config.json",
)
print("cached at:", path)

## Step 7 — Keep large files out of git

Use `.gitignore` for re-fetchable data, **Git LFS** to share weights via git, or **DVC** for reproducible, versioned datasets backed by S3/GCS. For this course, `.gitignore` is enough.